# Multi-Source Business Entity Resolution: Live Interactive Demo

This notebook demonstrates the end-to-end entity matching pipeline directly in your browser:
1. **Pre-processing & NFKD Unicode Normalization** (`src/normalize.py`)
2. **BM25-IDF Multi-Field Inverted Index Blocking** (`src/blocking.py`)
3. **11-Dimensional Pairwise Feature Extraction** (`src/features.py`)
4. **Trained LightGBM Classifier Prediction & Thresholding ($\tau^* = 0.74$)** (`src/model.py`)

In [ ]:
# 1. Install lightweight dependencies
!pip install rapidfuzz lightgbm pandas numpy -q

import sys
import os
import pickle
import pandas as pd
import numpy as np

# Add source directory to path
sys.path.append('code/business_entity_resolution/src')
from normalize import normalize_text
from blocking import CountryInvertedBlocker
from features import extract_pair_features, FEATURE_NAMES

In [ ]:
# 2. Load trained model and hyperparameter metadata
model_path = 'code/business_entity_resolution/models/matcher_model.pkl'
meta_path = 'code/business_entity_resolution/models/model_meta.pkl'

with open(model_path, 'rb') as f:
    model = pickle.load(f)
with open(meta_path, 'rb') as f:
    meta = pickle.load(f)

threshold = meta.get('optimal_threshold', 0.74)
print(f"Loaded LightGBM model successfully!")
print(f"Calibrated Optimal Decision Threshold (tau*): {threshold:.2f}")
print(f"Feature Count: {len(FEATURE_NAMES)}")

In [ ]:
# 3. Load Sample Multi-Source Records
df_s1 = pd.read_csv('demo_sample/sample_source1.tsv', sep='\t')
df_s2 = pd.read_csv('demo_sample/sample_source2.tsv', sep='\t')
df_s3 = pd.read_csv('demo_sample/sample_source3.tsv', sep='\t')
df_targets = pd.concat([df_s2, df_s3], ignore_index=True)

print(f"Sample Reference Entities (Source 1): {len(df_s1)}")
print(f"Sample Candidate Pool (Sources 2 & 3): {len(df_targets)}")
df_s1[['entity_id', 'business_name', 'business_address', 'country']].head(3)

In [ ]:
# 4. High-Speed Preprocessing
df_s1['norm_name'] = df_s1['business_name'].astype(str).apply(normalize_text)
df_s1['norm_addr'] = df_s1['business_address'].astype(str).apply(normalize_text)

df_targets['norm_name'] = df_targets['business_name'].astype(str).apply(normalize_text)
df_targets['norm_addr'] = df_targets['business_address'].astype(str).apply(normalize_text)

print("Sample Normalized Record:")
print("  Raw Name:       ", df_s1['business_name'].iloc[0])
print("  Normalized Name:", df_s1['norm_name'].iloc[0])
print("  Normalized Addr:", df_s1['norm_addr'].iloc[0])

In [ ]:
# 5. BM25-IDF Blocking & Candidate Retrieval
blocker = CountryInvertedBlocker(top_k=10, max_postings=1000, top_tokens_per_field=4)
blocker.fit_targets(
    target_ids=df_targets['entity_id'].values,
    target_norm_names=df_targets['norm_name'].tolist(),
    target_norm_addrs=df_targets['norm_addr'].tolist()
)

cands_dict, scores_dict = blocker.query_batch(
    query_ids=df_s1['entity_id'].tolist(),
    query_norm_names=df_s1['norm_name'].tolist(),
    query_norm_addrs=df_s1['norm_addr'].tolist()
)

print(f"Blocking Complete! Extracted candidates for {len(cands_dict)} reference entities.")

In [ ]:
# 6. 11-Dimensional Feature Extraction & Matcher Inference
results = []
tgt_map = df_targets.set_index('entity_id').to_dict('index')

for _, row in df_s1.iterrows():
    s1_id = row['entity_id']
    q_name = row['norm_name']
    q_addr = row['norm_addr']
    candidates = cands_dict.get(s1_id, [])
    cand_scores = scores_dict.get(s1_id, [])
    
    matched_ids = []
    for tid, b_score in zip(candidates, cand_scores):
        if tid in tgt_map:
            t_name = tgt_map[tid]['norm_name']
            t_addr = tgt_map[tid]['norm_addr']
            feats = extract_pair_features(q_name, q_addr, t_name, t_addr, b_score)
            prob = model.predict_proba([feats])[0, 1]
            if prob >= threshold:
                matched_ids.append((tid, prob))
                
    results.append({
        'Source1_ID': s1_id,
        'Business_Name': row['business_name'],
        'Address': row['business_address'],
        'Resolved_Matches': [m[0] for m in matched_ids],
        'Status': 'MATCHED' if matched_ids else 'SINGLETON'
    })

df_res = pd.DataFrame(results)
print("\n--- ENTITY RESOLUTION RESULTS ---")
df_res[['Source1_ID', 'Business_Name', 'Status', 'Resolved_Matches']].head(10)